###Task 3: Model Development 

In [2]:
#Load cleaned dataset
import pandas as pd

model_df = pd.read_csv("cleaned_dataset.csv")

In [3]:
model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25129 entries, 0 to 25128
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              25129 non-null  int64  
 1   gender          25129 non-null  str    
 2   car             25129 non-null  str    
 3   reality         25129 non-null  str    
 4   no_of_child     25129 non-null  int64  
 5   family_type     25129 non-null  str    
 6   house_type      25129 non-null  str    
 7   flag_mobil      25129 non-null  int64  
 8   work_phone      25129 non-null  int64  
 9   phone           25129 non-null  int64  
 10  e_mail          25129 non-null  int64  
 11  family_size     25129 non-null  float64
 12  begin_month     25129 non-null  int64  
 13  age             25129 non-null  int64  
 14  years_employed  25129 non-null  float64
 15  target          25129 non-null  int64  
 16  income          25129 non-null  float64
 17  income_type     25129 non-null  str    
 1

In [4]:
#data encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

categorical_cols = ['gender','car','reality','income_type','education_type','family_type','house_type']

for col in categorical_cols:
    model_df[col] = le.fit_transform(model_df[col])

In [6]:
#Feature Selection (X and y)
X = model_df.drop('target', axis=1)
y = model_df['target']

In [7]:
#Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
#Feature Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

###Model Comparison table creation

In [12]:
#Create result storage
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results = []

def evaluate_model(name, y_test, y_pred):
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0)
    })

In [13]:
#Logistic Regression - use SCALED data
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, class_weight='balanced')

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

evaluate_model("Logistic Regression (Balanced)", y_test, y_pred_lr)

In [14]:
#Random Forest (NO scaling)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

evaluate_model("Random Forest", y_test, y_pred_rf)

In [15]:
#Decision Tree (NO scaling)
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

evaluate_model("Decision Tree", y_test, y_pred_dt)

In [16]:
#ANN (use SCALED data)
from sklearn.neural_network import MLPClassifier

ann = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=42)

ann.fit(X_train_scaled, y_train)

y_pred_ann = ann.predict(X_test_scaled)

evaluate_model("ANN (MLP)", y_test, y_pred_ann)

In [19]:
results_df = pd.DataFrame(results)
results_df.sort_values(by='Recall', ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression (Balanced),0.639276,0.030601,0.589474,0.058182
2,Decision Tree,0.968166,0.150538,0.147368,0.148936
3,ANN (MLP),0.977318,0.297872,0.147368,0.197183
1,Random Forest,0.980899,0.473684,0.094737,0.157895


In [20]:
#Hyperparameter Tuning (Improved Logistic Regression)
from sklearn.linear_model import LogisticRegression

lr_improved = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    C=0.5,
    solver='liblinear'
)

lr_improved.fit(X_train_scaled, y_train)

y_pred_lr_imp = lr_improved.predict(X_test_scaled)

evaluate_model("Improved Logistic Regression", y_test, y_pred_lr_imp)